In [1]:
import os 
import numpy as np 
import pickle
import datetime as dt 

---

**Creating a stability dataset of the simulations**  
Label corresponds to whether a layout is statically stable. 

In [37]:
NUM_SAMPLES_PER_TRAJECTORY = 5  # all samples of some trajectory T have the same label 
SPLIT = 0.7

In [39]:
def assess_stability(arr):
    # TODO: Figure out how to assess the stability based on the trajectories of all items in a simulation. 
    #       If it boils down to comparing the beginning with the end, I need to consider that many items start in the air.
    #       If this is a problem, I might only be able to create this stability dataset once I implemented a way to create
    #       realistic layouts rather than just random ones.
    #       But in either case, I might have to incorporate some kind of "relaxation" phase or to incorporate some kind of 
    #       "relaxation" in the position delta. 
    #       Instead or additionally, I could simply look at non-vertical positional changes, so only xy-changes
    #       This way, I ignore the fact that items might start at an unreasonable and random height.
    
    delta = calculate_delta(arr, axes=(0,1))  # xy delta
    
    # 0.05 threshold makes sense at least for up to 20 items; still need to test it for more items
    return delta < 0.05

In [241]:
def calculate_delta(arr, t1=0, t2=-1, mean=True, axes=(0,1,2)):
    # arr is of shape (num_timesteps, num_items, 11)
    # axes: e.g. xy deltas would be axes=(0,1), full xyz delta would be axes=(0,1,2)
    # FIXME: does it make sense to use the mean? just because there are 10 more items that, say, don't touch any other items 
    #  and none of them move in the xy dimension doesn't make the other items more stable; but with the mean, this would drag
    #  down the overall delta
    a = arr[t1,:,axes].reshape(-1, len(axes))  # initial state
    b = arr[t2,:,axes].reshape(-1, len(axes))  # final state
    # Mean Euclidean distance between final and initial states
    delta = np.linalg.norm(b-a, axis=1).sum()
    if mean:
        delta /= len(a)
    return delta

In [307]:
deltas = []

min_simulation_id = None
for filename in os.listdir("cuboid_simulations"):
    if ".npy" in filename:
        v = np.load(open("cuboid_simulations/" + filename, "rb"))
        deltas.append(calculate_delta(v, axes=(0,1)))

deltas = np.array(deltas)
print(deltas.min())

0.019510018748173867


In [304]:
# Find simulation with minimum delta 

def find_minimum_delta_simulation(axes=(0,1)):
    deltas = []
    # min_simulation = None  # simulation corresponding to min_delta
    min_simulation_id = None
    for filename in os.listdir("cuboid_simulations"):
        if ".npy" in filename:
            v = np.load(open("cuboid_simulations/" + filename, "rb"))
            delta = calculate_delta(v, axes=axes)
            if len(deltas) > 0 and delta < min(deltas):
                # min_simulation = v 
                min_simulation_id = filename.replace(".npy", "")
            deltas.append(delta)
    deltas = np.array(deltas)
    print(deltas.min())
    return min_simulation_id

find_minimum_delta_simulation()

0.1041131818769085


'2023_01_20_01_57_08_107289'

In [94]:
num_simulations = len([filename for filename in os.listdir("cuboid_simulations") if ".npy" in filename])
num_train_simulations = int(SPLIT*num_simulations)
print(num_simulations, "total simulations")
print(num_train_simulations, "training simulations")

num_examples = num_simulations*NUM_SAMPLES_PER_TRAJECTORY
num_train_examples = num_train_simulations*NUM_SAMPLES_PER_TRAJECTORY
print(num_examples, "total examples")
print(num_train_examples, "training examples")

V = []
Y = []
for filename in os.listdir("cuboid_simulations"):
    if ".npy" in filename:
        v = np.load(open("cuboid_simulations/" + filename, "rb"))
        # 500+ samples for one item's trajectory in one particular simulation is probably too dense
        # I will sample NUM_SAMPLES_PER_TRAJECTORY entries, and try to have enough simulations and items 
        # to have a large and diverse dataset
        stability_score = assess_stability(v)
        y = [stability_score] * NUM_SAMPLES_PER_TRAJECTORY
        Y.extend(y)
        v = v[np.random.choice(range(len(v)), size=min(NUM_SAMPLES_PER_TRAJECTORY, len(v)), replace=False)]
        # Here I don't care about different timesteps. I simply want a list of 11-vectors
        # TODO: be able to deal with examples of varying number of items and timesteps
        #       e.g. by filling them up with "null items"
        #       but what would be even better (though I'm not sure how to do this yet), is to actually have them be empty
        #       and then later let the model learn an embedding corresponding to an [EMPTY] token
        #
        #
        #
        #
        V.extend(list(v))

V = np.array(V)
Y = np.array(Y)

V_train = V[:num_train_examples]
V_test = V[num_train_examples:]
y_train = Y[:num_train_examples]
y_test = Y[num_train_examples:]

# Normalize the data using statistics from the training set
mean, std = V_train.mean(), V_train.std()
V_train = (V_train-mean)/std
V_test = (V_test-mean)/std

# No longer needed
# del Y, V, mean, std

2000 total simulations
1400 training simulations
10000 total examples
7000 training examples


In [113]:
V_train.shape

(7000, 10, 11)

In [97]:
# """
# Store the whole dataset (before applying embeddings since I'm just using a random linear projection here for experimentation) 
# as a single pickled file

timestamp = str(dt.datetime.now()).replace(" ", "_").replace(":", "_").replace("-", "_").replace(".", "_")
pickle.dump(((V_train, y_train), (V_test, y_test)), open(f"datasets/stability/dataset_{timestamp}.pickle", "wb"))

# E.g. in Colab, read it like this:
# (V_train, y_train), (V_test, y_test) = pickle.load(open("datasets/stability/dataset_{timestamp}.pickle", "rb"))
# """;